
# NeuralRetail – Churn Prediction & SHAP Explainability
## AI-Powered Customer Retention Intelligence

### Project Phase
Customer Churn Analytics & Explainable AI

### Objectives
This notebook:
- Defines customer churn
- Creates churn prediction dataset
- Engineers churn-related features
- Trains machine learning models
- Evaluates churn performance
- Generates SHAP explainability
- Saves production-ready churn model

### Business Goal
Identify customers likely to stop purchasing and enable proactive retention strategies.

---


# 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score
)

from xgboost import XGBClassifier

import shap
import warnings
warnings.filterwarnings("ignore")


: 

# 2. Load Processed Dataset

In [ ]:

df = pd.read_csv("../data/processed/cleaned_retail_data.csv")

df.head()


In [ ]:

print("Dataset Shape:", df.shape)


# 3. Datetime Processing

In [ ]:

df['invoicedate'] = pd.to_datetime(df['invoicedate'])

print(df['invoicedate'].min())
print(df['invoicedate'].max())


# 4. Create Churn Target Variable

In [ ]:

# Reference date

snapshot_date = df['invoicedate'].max() + pd.Timedelta(days=1)

# Create RFM table

rfm = df.groupby('customerid').agg({
    'invoicedate': lambda x: (snapshot_date - x.max()).days,
    'invoiceno': 'nunique',
    'total_amount': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

rfm.head()


In [ ]:

# Define churn

# Customers inactive for more than 90 days = churned

rfm['Churn'] = np.where(rfm['Recency'] > 90, 1, 0)

rfm.head()



## Business Insight

Customers with high inactivity periods are considered churn-risk customers.
This enables predictive retention campaigns and customer lifecycle monitoring.


# 5. Feature Engineering

In [ ]:

# Additional features

rfm['AvgMonetaryPerPurchase'] = (
    rfm['Monetary'] / rfm['Frequency']
)

rfm['CustomerValueScore'] = (
    rfm['Frequency'] * rfm['Monetary']
)

rfm.head()


In [ ]:

rfm.describe()


# 6. Churn Distribution

In [ ]:

plt.figure(figsize=(6,4))

sns.countplot(x=rfm['Churn'])

plt.title("Churn Distribution")

plt.show()



## Business Insight

Understanding churn distribution is critical for:
- Customer retention planning
- Campaign optimization
- Revenue protection


# 7. Feature Selection

In [ ]:

features = [
    'Recency',
    'Frequency',
    'Monetary',
    'AvgMonetaryPerPurchase',
    'CustomerValueScore'
]

X = rfm[features]
y = rfm['Churn']

print(X.head())


# 8. Train-Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


# 9. Train XGBoost Churn Model

In [ ]:

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

print("XGBoost model trained successfully.")


# 10. Model Prediction

In [ ]:

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:,1]


# 11. Model Evaluation

In [ ]:

accuracy = accuracy_score(y_test, y_pred)

auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {accuracy:.4f}")
print(f"AUC-ROC: {auc:.4f}")


In [ ]:

print(classification_report(y_test, y_pred))


In [ ]:

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


In [ ]:

# ROC Curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8,6))

plt.plot(fpr, tpr)

plt.plot([0,1], [0,1], linestyle='--')

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.show()



## Evaluation Interpretation

AUC-ROC measures the model's ability to distinguish churned vs non-churned customers.

Project Target:
- AUC-ROC ≥ 0.90


# 12. SHAP Explainability

In [ ]:

# SHAP Explainer

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(X_test)

print("SHAP values generated successfully.")


In [ ]:

# SHAP Summary Plot

shap.summary_plot(shap_values, X_test)


In [ ]:

# SHAP Beeswarm Plot

shap.plots.beeswarm(
    shap.Explanation(
        values=shap_values,
        data=X_test.values,
        feature_names=X_test.columns
    )
)


In [ ]:

# SHAP Waterfall Plot

sample_index = 0

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_index],
        base_values=explainer.expected_value,
        data=X_test.iloc[sample_index],
        feature_names=X_test.columns
    )
)



## Business Insight

SHAP explainability identifies:
- Why customers are likely to churn
- Which features influence churn most
- Which retention actions should be prioritized

This transforms the model from a black-box system into explainable business intelligence.


# 13. Save Churn Model

In [ ]:

import os
import pickle

os.makedirs("../output/models", exist_ok=True)

# Save model

with open("../output/models/xgboost_churn_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Churn model saved successfully.")


# 14. Save Churn Dataset

In [ ]:

os.makedirs("../data/processed", exist_ok=True)

churn_df = rfm
rfm.to_csv(
    "../data/processed/churn_dataset.csv",
    index=False
)

print("Churn dataset saved successfully.")


In [ ]:
# ADD SEGMENT + CLUSTER FEATURES
featured_df = pd.read_csv("../data/processed/featured_data.csv")

# Add Segment
if 'Segment' in featured_df.columns:
    churn_df['Segment'] = featured_df['Segment']

# Add Cluster
if 'Cluster' in featured_df.columns:
    churn_df['Cluster'] = featured_df['Cluster']

# Reorder columns
final_cols = [
    'CustomerID',
    'Recency',
    'Frequency',
    'Monetary',
    'AvgMonetaryPerPurchase',
    'CustomerValueScore',
    'Segment',
    'Cluster',
    'Churn'
]

final_cols = [c for c in final_cols if c in churn_df.columns]

churn_df = churn_df[final_cols]

# Save again
churn_df.to_csv(
    '../data/processed/churn_dataset.csv',
    index=False
)

print(churn_df.columns.tolist())
print(churn_df.shape)

In [ ]:
# =========================================================
# CREATE CUSTOMER EXPORT DATA
# =========================================================

customer_df = X_test.copy()

customer_df["CustomerID"] = rfm.loc[
    X_test.index,
    "CustomerID"
].values

# REAL PROBABILITIES
customer_df["Churn_Probability"] = y_prob

# ADD BUSINESS FEATURES
customer_df["Segment"] = rfm.loc[
    X_test.index,
    "Segment"
].values

customer_df["Recency"] = rfm.loc[
    X_test.index,
    "Recency"
].values

customer_df["Frequency"] = rfm.loc[
    X_test.index,
    "Frequency"
].values

customer_df["Monetary"] = rfm.loc[
    X_test.index,
    "Monetary"
].values

print(customer_df.head())

In [ ]:
# =========================================================
# Export Churn Predictions
# =========================================================

churn_export = customer_df[[

    'CustomerID',

    'Churn_Probability',

    'Segment',

    'Recency',

    'Frequency',

    'Monetary'
]]

churn_export.to_csv(

    "../dashboard/data/churn_predictions.csv",

    index=False
)

print("✅ churn_predictions.csv exported successfully")


# Final Conclusion

This notebook successfully:
- Defined customer churn
- Engineered churn-related features
- Trained XGBoost churn prediction model
- Evaluated classification performance
- Generated SHAP explainability visualizations
- Saved production-ready model artifacts

### Business Value
The churn engine enables:
- Proactive retention campaigns
- Revenue protection
- Customer intelligence
- Personalized marketing actions


## NeuralRetail – Amdox Technologies
